In [1]:
%pip install pdfminer.six

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pdfminer.high_level import extract_text
import re

# Step 1: Extract text with pdfminer (better Unicode support)
file_path = "docs\wiyanjana.pdf"
text = extract_text(file_path)

# Step 2: Check if text is extracted
print("Preview of extracted text:\n", text[:500])  # See what came out

# Step 3: Split by each “ක්‍රියාකාරකම” section
sections = re.split(r"ක්‍රියාකාරකම[-–]\s*\d+", text)
section_titles = re.findall(r"ක්‍රියාකාරකම[-–]\s*\d+", text)

# Step 4: Tokenize each section into Sinhala words
tokenized_sections = {}
for i, section in enumerate(sections[1:]):  # skip any preface text
    words = re.findall(r"[අ-ෆ]+", section)
    tokenized_sections[section_titles[i]] = words

# Step 5: Display results
for title, tokens in tokenized_sections.items():
    print(f"\n{title}:")
    print(tokens)


Preview of extracted text:
 ක්‍රියාකාරකම-01 

01 

02 

03 

04 

05 

06 

07 

08 

09 

10 

11 

12 

13 

14 

15 

පස් 

බස් 

පපොල් 

පබොල් 

පිල 

පය 

පත් 

ටයි 

කල 

කැල 

කස 

බිල 

බය 

බත් 

ඩයි 

ගල 

ගැල 

ගස 

කලන 

ගලන 

කර 

කත් 

කල 

ෆෑන් 

පෆ්ල් 

දර 

දත් 

දල 

වෑන් 

පේල් 

ක්‍රියාකාරකම-02 

01 

02 

03 

කප 

අප 

ඔප 

කබ 

අබ 

ඔබ 

 
 
 
 
 
04 

05 

06 

07 

08 

09 

10 

11 

12 

13 

14 

15 

ඇප 

වැට 

කට 

අට 

ඇබ 

වැඩ 

කඩ 

අඩ 

ප ොට 

ප ොඞ 

පකෝටු 

පකෝඩු 

අත 

කත 

අද 

කද 

ම 

ක්‍රියාකාරකම-01:
['පස', 'බස', 'පප', 'ල', 'පබ', 'ල', 'ප', 'ල', 'පය', 'පත', 'ටය', 'කල', 'ක', 'ල', 'කස', 'බ', 'ල', 'බය', 'බත', 'ඩය', 'ගල', 'ග', 'ල', 'ගස', 'කලන', 'ගලන', 'කර', 'කත', 'කල', 'ෆ', 'න', 'පෆ', 'ල', 'දර', 'දත', 'දල', 'ව', 'න', 'ප', 'ල']

ක්‍රියාකාරකම-02:
['කප', 'අප', 'ඔප', 'කබ', 'අබ', 'ඔබ', 'ඇප', 'ව', 'ට', 'කට', 'අට', 'ඇබ', 'ව', 'ඩ', 'කඩ', 'අඩ', 'ප', 'ට', 'ප', 'ඞ', 'පක', 'ට', 'පක', 'ඩ', 'අත', 'කත', 'අද', 'කද', 'ම', 'ත', 'ම', 'ද', 'වත', 'පත', 'ත', 'වද', 'පද', '

In [3]:
# Imports and helpers
import re
import json
from pathlib import Path
from docx import Document
from docx.text.paragraph import Paragraph
from docx.table import Table
from IPython.display import JSON, display

# Regex patterns
HEADER_RE = re.compile(r"^(ක්‍රියාකාරකම\s*[-–]?\s*\d+)\s*$", re.IGNORECASE)
NUMBER_RE = re.compile(r"^(\d{1,2})\b(?:[.)\-]?\s*)(.*)$")
SINHALA_TOKEN_RE = re.compile(r"[\u0D80-\u0DFF]+(?:[\-–—]?[\u0D80-\u0DFF]+)*")

def iter_doc_blocks(doc: Document):
    """
    Yield text blocks (strings) in the order they appear in the document body,
    covering both paragraphs and table cells.
    Each yielded block is stripped.
    """
    body = doc.element.body
    for child in body.iterchildren():
        # paragraph
        if child.tag.endswith('}p'):
            para = Paragraph(child, doc)
            text = para.text.strip()
            if text:
                yield text
        # table
        elif child.tag.endswith('}tbl'):
            tbl = Table(child, doc)
            for r in tbl.rows:
                for c in r.cells:
                    # split lines inside a cell and yield each non-empty line as a block
                    for line in c.text.splitlines():
                        line = line.strip()
                        if line:
                            yield line

def extract_tokens_from_text(text):
    """Return list of Sinhala tokens from text (preserve order)."""
    return [m.group(0) for m in SINHALA_TOKEN_RE.finditer(text)]


In [4]:
# Parsing function (core logic)
def parse_blocks(blocks):
    """
    Given a list of ordered text blocks (paragraphs and table cells),
    return parsed structure mapping headers -> { items: [...], tokens: [...] }.
    """
    result = {}
    current_header = None
    i = 0
    n = len(blocks)
    while i < n:
        block = blocks[i].strip()
        # detect header
        h = HEADER_RE.match(block)
        if h:
            # normalize header to remove internal whitespace so keys look like "ක්‍රියාකාරකම-01"
            header_key = re.sub(r"\s+", "", h.group(1))
            current_header = header_key
            if current_header not in result:
                result[current_header] = {"items": [], "tokens": []}
            i += 1
            continue

        # if inside a section, look for numbered entry
        if current_header:
            m = NUMBER_RE.match(block)
            if m:
                idx = m.group(1).zfill(2)
                remainder = m.group(2).strip()
                words_line = remainder

                # if number line has no words, look ahead for next 1-2 blocks as the word(s)
                if not words_line:
                    collected = []
                    j = i + 1
                    while j < n and len(collected) < 2:
                        nxt = blocks[j].strip()
                        # if next block starts a new header or number, stop
                        if HEADER_RE.match(nxt) or NUMBER_RE.match(nxt):
                            break
                        if nxt:
                            collected.append(nxt)
                        j += 1
                    words_line = " ".join(collected).strip()
                    i = j  # advance past the blocks we consumed
                else:
                    i += 1

                # extract Sinhala tokens (preserve exact strings)
                tokens = extract_tokens_from_text(words_line)
                if len(tokens) >= 2:
                    words_pair = tokens[:2]
                elif len(tokens) == 1:
                    words_pair = [tokens[0]]
                else:
                    # fallback: maybe the block itself contains tokens
                    fb = extract_tokens_from_text(block)
                    words_pair = fb[:2] if fb else []

                result[current_header]["items"].append({"index": idx, "words": words_pair})
                for t in words_pair:
                    result[current_header]["tokens"].append(t)
                continue

        # not a header/number: move on
        i += 1

    return result


In [5]:
# Main execution cell: load docx, parse, save JSON, and display a preview
DOCX_PATH = Path("docs\wiyanjana.docx")  # change path if needed
OUT_JSON = Path("wiyanjana_from_docx_notebook.json")

if not DOCX_PATH.exists():
    raise FileNotFoundError(f"{DOCX_PATH} not found. Upload wiyanjana.docx to the notebook folder or change DOCX_PATH.")

doc = Document(DOCX_PATH)
blocks = [b for b in iter_doc_blocks(doc) if b.strip() != ""]

print(f"Found {len(blocks)} text blocks (paragraph lines & table cells).")
# Optional: show first 30 blocks for debugging
# for k,bl in enumerate(blocks[:30],1): print(f"{k:02d} | {bl}")

parsed = parse_blocks(blocks)

if not parsed:
    print("No sections parsed. Showing first 50 blocks for debugging:")
    for k, b in enumerate(blocks[:50], start=1):
        print(f"{k:02d} | {b!r}")
else:
    # Save JSON
    OUT_JSON.write_text(json.dumps(parsed, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Saved JSON to:", OUT_JSON.resolve())
    # Display a readable preview in the notebook (first section)
    first_key = next(iter(parsed))
    print("\nPreview (first section):", first_key)
    display(JSON(parsed[first_key]))


Found 281 text blocks (paragraph lines & table cells).
Saved JSON to: C:\Users\94772\Desktop\HearingProject\wiyanjana_from_docx_notebook.json

Preview (first section): ක්‍රියාකාරකම-01


<IPython.core.display.JSON object>

In [6]:
# Main execution cell: load docx, parse, save JSON, and display a preview
DOCX_PATH = Path("docs\wiyanjana.docx")  # change path if needed
OUT_JSON = Path("wiyanjana_transliterated.json")

if not DOCX_PATH.exists():
    raise FileNotFoundError(f"{DOCX_PATH} not found. Upload wiyanjana.docx to the notebook folder or change DOCX_PATH.")

doc = Document(DOCX_PATH)
blocks = [b for b in iter_doc_blocks(doc) if b.strip() != ""]

print(f"Found {len(blocks)} text blocks (paragraph lines & table cells).")
# Optional: show first 30 blocks for debugging
# for k,bl in enumerate(blocks[:30],1): print(f"{k:02d} | {bl}")

parsed = parse_blocks(blocks)

if not parsed:
    print("No sections parsed. Showing first 50 blocks for debugging:")
    for k, b in enumerate(blocks[:50], start=1):
        print(f"{k:02d} | {b!r}")
else:
    # Save JSON
    OUT_JSON.write_text(json.dumps(parsed, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Saved JSON to:", OUT_JSON.resolve())
    # Display a readable preview in the notebook (first section)
    first_key = next(iter(parsed))
    print("\nPreview (first section):", first_key)
    display(JSON(parsed[first_key]))


Found 281 text blocks (paragraph lines & table cells).
Saved JSON to: C:\Users\94772\Desktop\HearingProject\wiyanjana_transliterated.json

Preview (first section): ක්‍රියාකාරකම-01


<IPython.core.display.JSON object>

In [7]:
# Improved Sinhala -> Singlish transliteration (notebook cell)
import json
from pathlib import Path
import unicodedata

IN_JSON = Path("wiyanjana_transliterated.json")
OUT_JSON = Path("Wiyanjana_translit_improved.json")

if not IN_JSON.exists():
    raise FileNotFoundError(f"Input JSON not found: {IN_JSON}")

# --- mapping: consonant -> base latin consonant (no trailing vowel) ---
CONS = {
    "ක":"k","ඛ":"kh","ග":"g","ඝ":"gh","ඟ":"ng",
    "ච":"ch","ඡ":"chh","ජ":"j","ඣ":"jh","ඤ":"ny",
    "ට":"t","ඨ":"th","ඩ":"d","ඪ":"dh","ණ":"n",
    "ත":"t","ථ":"th","ද":"d","ධ":"dh","න":"n",
    "ප":"p","ඵ":"ph","බ":"b","භ":"bh","ම":"m",
    "ය":"y","ර":"r","ල":"l","ව":"v","ශ":"sh",
    "ෂ":"sh","ස":"s","හ":"h","ළ":"l","ෆ":"f",
    # extras
    "ඥ":"gn","ඦ":"gn"
}

# independent vowels
INDEP_V = {
    "අ":"a","ආ":"aa","ඇ":"ae","ඈ":"aae",
    "ඉ":"i","ඊ":"ii","උ":"u","ඌ":"uu",
    "එ":"e","ඒ":"ee","ඔ":"o","ඕ":"oo",
    "ඓ":"ai","ඖ":"au"
}

# vowel signs (matras) that follow consonants
V_SIGN = {
    "ා":"aa",  # long a
    "ි":"i",   # i
    "ී":"ii",  # long i
    "ු":"u",   # u
    "ූ":"uu",  # long u
    "ෙ":"e",   # e
    "ේ":"ee",  # long e
    "ෛ":"ai",
    "ො":"o",
    "ෝ":"oo",
    "ෞ":"au",
    "ෘ":"ru",  # approximation
    "ෲ":"ruu"
}

DIAC = {
    "ං":"n",   # anusvara
    "ඃ":"h"
}

VIRAMA = "්"

# sets for fast checks
CONS_SET = set(CONS.keys())
INDEP_SET = set(INDEP_V.keys())
V_SIGN_SET = set(V_SIGN.keys())
DIAC_SET = set(DIAC.keys())

def normalize_text(s: str) -> str:
    # NFC normal form
    return unicodedata.normalize("NFC", s)

def transliterate_word(word: str, cap_first: bool = True) -> str:
    w = normalize_text(word)
    i = 0
    n = len(w)
    out = []

    while i < n:
        ch = w[i]

        # independent vowel
        if ch in INDEP_SET:
            out.append(INDEP_V[ch])
            i += 1
            continue

        # consonant + possible sequences
        if ch in CONS_SET:
            # gemination: C + virama + same C  -> double consonant
            # e.g., ල + ෍ (virama) + ල  (i.e. ල්ලා)
            if i+2 < n and w[i+1] == VIRAMA and w[i+2] == ch:
                base = CONS[ch]  # base consonant translit
                # consume C, virama, second C
                i += 3
                # after the doubled consonant may come a vowel sign or diacritic
                vowel_part = ""
                while i < n and (w[i] in V_SIGN_SET or w[i] in DIAC_SET):
                    if w[i] in V_SIGN_SET:
                        vowel_part = V_SIGN[w[i]]
                    elif w[i] in DIAC_SET:
                        vowel_part += DIAC[w[i]]
                    i += 1
                # if no explicit vowel sign -> implicit 'a'
                if vowel_part == "":
                    vowel_part = "a"
                out.append(base + base + vowel_part)  # doubled consonant + vowel
                continue

            # normal consonant handling
            base = CONS[ch]
            i += 1
            vowel_part = ""
            had_vowel_sign = False
            # if next is virama -> pure consonant (no vowel)
            if i < n and w[i] == VIRAMA:
                # consume virama, produce base (no vowel)
                i += 1
                out.append(base)  # no trailing vowel
                continue

            # collect vowel signs/diacritics immediately following consonant
            while i < n and (w[i] in V_SIGN_SET or w[i] in DIAC_SET):
                if w[i] in V_SIGN_SET:
                    vowel_part = V_SIGN[w[i]]
                    had_vowel_sign = True
                elif w[i] in DIAC_SET:
                    vowel_part += DIAC[w[i]]
                i += 1

            # if no vowel sign, implicit 'a'
            if vowel_part == "":
                vowel_part = "a"
            out.append(base + vowel_part)
            continue

        # vowel sign or diacritic alone (rare) - map if possible
        if ch in V_SIGN_SET:
            out.append(V_SIGN[ch])
            i += 1
            continue
        if ch in DIAC_SET:
            out.append(DIAC[ch])
            i += 1
            continue

        # anything else (punctuation, ASCII) passes through
        out.append(ch)
        i += 1

    result = "".join(out)
    if cap_first and result:
        result = result[0].upper() + result[1:]
    return result

# quick test set with hearing and language learning related words
test_words = [
    "කන", "ඇහැ", "අහන්න", "කියන්න", "දැනුම", "උගන්න",
    "ශබ්ද", "වචන", "අකුරු", "හඬ", "භාෂා", "කථාව",
    "පාඩම", "අර්ථ", "පිළිතුර"
]
print("Test transliterations:")
for tw in test_words:
    print(tw, "->", transliterate_word(tw))

# load input JSON, transliterate and save
data = json.loads(IN_JSON.read_text(encoding="utf-8"))
outd = {}
for sec, payload in data.items():
    outd[sec] = {"items": [], "tokens": []}
    for it in payload.get("items", []):
        idx = it.get("index", "")
        words = it.get("words", [])
        twords = [transliterate_word(w, cap_first=True) for w in words]
        outd[sec]["items"].append({"index": idx, "words": twords})
        outd[sec]["tokens"].extend(twords)

OUT_JSON.write_text(json.dumps(outd, ensure_ascii=False, indent=2), encoding="utf-8")
print("\nWrote:", OUT_JSON.resolve())

Test transliterations:
කන -> Kana
ඇහැ -> Aehaැ
අහන්න -> Ahanna
කියන්න -> Kiyanna
දැනුම -> Daැnuma
උගන්න -> Uganna
ශබ්ද -> Shabda
වචන -> Vachana
අකුරු -> Akuru
හඬ -> Haඬ
භාෂා -> Bhaashaa
කථාව -> Kathaava
පාඩම -> Paadama
අර්ථ -> Artha
පිළිතුර -> Pilitura

Wrote: C:\Users\94772\Desktop\HearingProject\Wiyanjana_translit_improved.json


In [8]:
# Notebook cell: extract changing consonants between word pairs
import json, csv, unicodedata, re
from pathlib import Path
from typing import Tuple, List
from unidecode import unidecode

IN_JSON = Path("wiyanjana_from_docx_notebook.json")
OUT_CSV = Path("wiyanjana_words_consonants_firstletter.csv")
OUT_JSON = Path("wiyanjana_words_consonants_firstletter.json")

# Audio path configuration
AUDIO_BASE_PATH = Path("hearing_project_v0.01/wiyanjana_audio_cloud")

# consonant mappings with their base forms
CONS = {
    "ක":"k","ඛ":"kh","ග":"g","ඝ":"gh","ඟ":"ng",
    "ච":"ch","ඡ":"chh","ජ":"j","ඣ":"jh","ඤ":"ny",
    "ට":"t","ඨ":"th","ඩ":"d","ඪ":"dh","ණ":"n",
    "ත":"t","ථ":"th","ද":"d","ධ":"dh","න":"n",
    "ප":"p","ඵ":"ph","බ":"b","භ":"bh","ම":"m",
    "ය":"y","ර":"r","ල":"l","ව":"v","ශ":"sh",
    "ෂ":"sh","ස":"s","හ":"h","ළ":"l","ෆ":"f",
    "ඥ":"gn","ඦ":"gn"
}

# independent vowels (needed for parsing)
INDEP_VOWELS = {
    "අ":"a","ආ":"aa","ඇ":"ae","ඈ":"aae",
    "ඉ":"i","ඊ":"ii","උ":"u","ඌ":"uu",
    "එ":"e","ඒ":"ee","ඔ":"o","ඕ":"oo",
    "ඓ":"ai","ඖ":"au"
}

# Vowel signs and diacritics (needed for parsing)
V_SIGN = {
    "ා":"aa", "ි":"i", "ී":"ii", "ු":"u", "ූ":"uu",
    "ෙ":"e", "ේ":"ee", "ෛ":"ai", "ො":"o", "ෝ":"oo",
    "ෞ":"au", "ෘ":"ru", "ෲ":"ruu"
}

DIAC = {
    "ං":"n",   # anusvara
    "ඃ":"h"    # visarga
}

VIRAMA = "්"

# sets for fast checks
INDEP_SET = set(INDEP_VOWELS.keys())
CONS_SET = set(CONS.keys())
V_SIGN_SET = set(V_SIGN.keys())
DIAC_SET = set(DIAC.keys())

def normalize_text(s: str) -> str:
    return unicodedata.normalize("NFC", s)

def extract_consonants(word: str) -> List[str]:
    """Extract all consonants from a word in order."""
    w = normalize_text(word)
    i = 0
    n = len(w)
    consonants = []
    
    while i < n:
        ch = w[i]
        
        # Skip independent vowels
        if ch in INDEP_SET:
            i += 1
            continue
            
        # Found a consonant
        if ch in CONS_SET:
            consonants.append(ch)
            i += 1
            # Skip any following vowel signs or diacritics
            while i < n and (w[i] in V_SIGN_SET or w[i] in DIAC_SET or w[i] == VIRAMA):
                i += 1
            continue
            
        # Skip other characters
        i += 1
        
    return consonants

def find_changing_consonant(word1: str, word2: str) -> Tuple[str, str]:
    """Find the consonant that changes between two words."""
    if not word1 or not word2:
        return "", ""
        
    cons1 = extract_consonants(word1)
    cons2 = extract_consonants(word2)
    
    # Compare consonants at each position
    min_len = min(len(cons1), len(cons2))
    for i in range(min_len):
        if cons1[i] != cons2[i]:
            return cons1[i], cons2[i]
    
    # If no difference found in common positions, check if one word has extra consonants
    if len(cons1) != len(cons2):
        if len(cons1) > len(cons2):
            return cons1[min_len], ""
        else:
            return "", cons2[min_len]
            
    return "", ""

def transliterate_word(word: str, cap_first: bool = True) -> str:
    """Convert Sinhala word to Singlish transliteration."""
    w = normalize_text(word)
    i = 0
    n = len(w)
    out = []

    while i < n:
        ch = w[i]

        # independent vowel
        if ch in INDEP_SET:
            out.append(INDEP_VOWELS[ch])
            i += 1
            continue

        # consonant + possible sequences
        if ch in CONS_SET:
            # gemination: C + virama + same C  -> double consonant
            if i+2 < n and w[i+1] == VIRAMA and w[i+2] == ch:
                base = CONS[ch]
                i += 3
                vowel_part = ""
                while i < n and (w[i] in V_SIGN_SET or w[i] in DIAC_SET):
                    if w[i] in V_SIGN_SET:
                        vowel_part = V_SIGN[w[i]]
                    elif w[i] in DIAC_SET:
                        vowel_part += DIAC[w[i]]
                    i += 1
                if vowel_part == "":
                    vowel_part = "a"
                out.append(base + base + vowel_part)
                continue

            # normal consonant handling
            base = CONS[ch]
            i += 1
            vowel_part = ""
            if i < n and w[i] == VIRAMA:
                i += 1
                out.append(base)
                continue

            while i < n and (w[i] in V_SIGN_SET or w[i] in DIAC_SET):
                if w[i] in V_SIGN_SET:
                    vowel_part = V_SIGN[w[i]]
                elif w[i] in DIAC_SET:
                    vowel_part += DIAC[w[i]]
                i += 1

            if vowel_part == "":
                vowel_part = "a"
            out.append(base + vowel_part)
            continue

        # vowel sign or diacritic alone
        if ch in V_SIGN_SET:
            out.append(V_SIGN[ch])
            i += 1
            continue
        if ch in DIAC_SET:
            out.append(DIAC[ch])
            i += 1
            continue

        # anything else passes through
        out.append(ch)
        i += 1

    result = "".join(out)
    if cap_first and result:
        result = result[0].upper() + result[1:]
    return result

def safe_unicode_filename(s: str) -> str:
    """Create a safe filename from a string."""
    s = s.strip()
    s = re.sub(r"[\\/:\*\?\"<>\|]", "_", s)
    s = re.sub(r"\s+", "_", s)
    return s[:60]

def ascii_translit(s: str) -> str:
    """Create ASCII transliteration."""
    return re.sub(r"[^\w\-_.]", "", unidecode(s)) or "word"

def get_audio_path(section: str, idx: str, word: str) -> str:
    """Generate the audio file path for a word."""
    uni_name = safe_unicode_filename(word)
    ascii_name = ascii_translit(word)
    return str(AUDIO_BASE_PATH / section / f"{idx}_{uni_name}_{ascii_name}.mp3")

# --- build rows ---
if not IN_JSON.exists():
    raise FileNotFoundError(f"Input JSON not found: {IN_JSON}")

data = json.loads(IN_JSON.read_text(encoding="utf-8"))
rows = []

for section, payload in data.items():
    for it in payload.get("items", []):
        idx = it.get("index", "")
        words = it.get("words", [])
        
        # We need pairs of words
        if len(words) >= 2:
            word1, word2 = words[:2]
            changing_cons1, changing_cons2 = find_changing_consonant(word1, word2)
            
            # Create entry for first word
            if word1:
                audio_path1 = get_audio_path(section, str(idx).zfill(2), word1)
                rows.append({
                    "section": section,
                    "index": idx,
                    "sinhala_word": word1,
                    "singlish_word": transliterate_word(word1),
                    "changing_consonant_character": changing_cons1,
                    "changing_consonant_name": CONS.get(changing_cons1, ""),
                    "audio_path": audio_path1
                })
            
            # Create entry for second word
            if word2:
                audio_path2 = get_audio_path(section, str(idx).zfill(2), word2)
                rows.append({
                    "section": section,
                    "index": idx,
                    "sinhala_word": word2,
                    "singlish_word": transliterate_word(word2),
                    "changing_consonant_character": changing_cons2,
                    "changing_consonant_name": CONS.get(changing_cons2, ""),
                    "audio_path": audio_path2
                })

# --- write CSV and JSON ---
fieldnames = [
    "section", "index", "sinhala_word", "singlish_word",
    "changing_consonant_character", "changing_consonant_name",
    "audio_path"
]

with OUT_CSV.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

OUT_JSON.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote:", OUT_CSV.resolve())
print("Wrote:", OUT_JSON.resolve())

# quick preview of output structure
print("\nExample outputs:")
for r in rows[:4]:
    print("\nSample row:", r)

Wrote: C:\Users\94772\Desktop\HearingProject\wiyanjana_words_consonants_firstletter.csv
Wrote: C:\Users\94772\Desktop\HearingProject\wiyanjana_words_consonants_firstletter.json

Example outputs:

Sample row: {'section': 'ක්\u200dරියාකාරකම-01', 'index': '01', 'sinhala_word': 'පස්', 'singlish_word': 'Pas', 'changing_consonant_character': 'ප', 'changing_consonant_name': 'p', 'audio_path': 'hearing_project_v0.01\\wiyanjana_audio_cloud\\ක්\u200dරියාකාරකම-01\\01_පස්_ps.mp3'}

Sample row: {'section': 'ක්\u200dරියාකාරකම-01', 'index': '01', 'sinhala_word': 'බස්', 'singlish_word': 'Bas', 'changing_consonant_character': 'බ', 'changing_consonant_name': 'b', 'audio_path': 'hearing_project_v0.01\\wiyanjana_audio_cloud\\ක්\u200dරියාකාරකම-01\\01_බස්_bs.mp3'}

Sample row: {'section': 'ක්\u200dරියාකාරකම-01', 'index': '02', 'sinhala_word': 'පොල්', 'singlish_word': 'Pol', 'changing_consonant_character': 'ප', 'changing_consonant_name': 'p', 'audio_path': 'hearing_project_v0.01\\wiyanjana_audio_cloud\\ක්\u2

In [9]:
# SINGLE NOTEBOOK CELL: Generate high-quality Sinhala audio per-word, sectioned folders.
# - Uses Google Cloud TTS (WaveNet) if GOOGLE_APPLICATION_CREDENTIALS is set.
# - Falls back to gTTS + pydub if not.
# - Post-processes with pydub: slow down slightly + increase volume.
# - Allows pronunciation overrides for words that are unclear.
#
# Install dependencies first (run in a notebook cell if needed):
# !pip install google-cloud-texttospeech gTTS pydub tqdm unidecode

import os, json, io, re
from pathlib import Path
from tqdm import tqdm
from unidecode import unidecode
from pydub import AudioSegment

# Try to import Google Cloud TTS
USE_GOOGLE_CLOUD = False
try:
    from google.cloud import texttospeech
    # If GOOGLE_APPLICATION_CREDENTIALS env var present, enable cloud usage
    if os.environ.get("GOOGLE_APPLICATION_CREDENTIALS"):
        USE_GOOGLE_CLOUD = True
except Exception:
    USE_GOOGLE_CLOUD = False

# Fallback gTTS
from gtts import gTTS

# ---------- Config ----------
INPUT_JSON = Path("wiyanjana_words_consonants_firstletter.json")   # change if your JSON has a different name
OUTPUT_ROOT = Path("hearing_project_v0.01/wiyanjana_audio_cloud")  # top-level output folder
LANG = "si"  # for gTTS fallback
# Post-processing
SLOW_FACTOR = 1.15       # 1.15 => ~15% slower (tweak to taste)
VOLUME_GAIN_DB = 5       # increase volume by +5 dB
# Google Cloud TTS params (only used if USE_GOOGLE_CLOUD True)
GCP_VOICE_NAME = "si-LK-Wavenet-A"   # example; adjust if needed or list available voices
GCP_AUDIO_ENCODING = texttospeech.AudioEncoding.MP3 if USE_GOOGLE_CLOUD else None
GCP_SPEAKING_RATE = 0.95   # 0.9-1.0 = slightly slower; fine tune
GCP_PITCH = 0.0            # adjust if voice sounds too low/high

# Pronunciation overrides: map original Sinhala word -> alternate text to send to TTS
# Use this to manually fix words with unclear pronunciation.
# Example: "පිනා": "පි-නා" or a small transliteration/phonetic form; try different forms until pronunciation is good.
pronunciation_overrides = {
    # "පිනා": "පි-නා",
    # "කජු": "ක-ජු",
    # add problematic words here and experiment
}

# ---------- Helpers ----------
def safe_unicode_filename(s: str) -> str:
    s = s.strip()
    s = re.sub(r"[\\/:\*\?\"<>\|]", "_", s)
    s = re.sub(r"\s+", "_", s)
    return s[:60]

def ascii_translit(s: str) -> str:
    return re.sub(r"[^\w\-_.]", "", unidecode(s)) or "word"

def change_speed(sound: AudioSegment, factor: float) -> AudioSegment:
    """Return a copy of sound slowed by `factor` (factor >1 -> slower)."""
    # pydub change via frame rate trick
    new_frame_rate = int(sound.frame_rate / factor)
    slowed = sound._spawn(sound.raw_data, overrides={"frame_rate": new_frame_rate})
    return slowed.set_frame_rate(sound.frame_rate)

def save_audiosegment_to_mp3(audio: AudioSegment, path: Path, bitrate="192k"):
    audio.export(str(path), format="mp3", bitrate=bitrate)

# ---------- Load input JSON ----------
if not INPUT_JSON.exists():
    raise FileNotFoundError(f"Input JSON not found: {INPUT_JSON.resolve()}")

records = json.loads(INPUT_JSON.read_text(encoding="utf-8"))

# Build grouping by section -> list of items (preserve given order)
from collections import OrderedDict
sections = OrderedDict()
for r in records:
    sec = r.get("section", "unknown")
    sections.setdefault(sec, []).append(r)

# Make output structure
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# If using Google Cloud TTS client, initialize
gcp_client = None
if USE_GOOGLE_CLOUD:
    try:
        gcp_client = texttospeech.TextToSpeechClient()
    except Exception as e:
        print("Google Cloud TTS import/initialization failed:", e)
        USE_GOOGLE_CLOUD = False

# ---------- Generation loop ----------
total_items = sum(len(v) for v in sections.values())
pbar = tqdm(total=total_items, desc="Generating audio")

for sec, items in sections.items():
    sec_folder = OUTPUT_ROOT / sec
    sec_folder.mkdir(parents=True, exist_ok=True)

    for item in items:
        idx = str(item.get("index", "")).zfill(2)
        word = item.get("sinhala_word", "").strip()
        if not word:
            pbar.update(1)
            continue

        # Use override text if provided
        tts_text = pronunciation_overrides.get(word, word)

        uni_name = safe_unicode_filename(word)
        ascii_name = ascii_translit(word)
        out_filename = f"{idx}_{uni_name}_{ascii_name}.mp3"
        out_path = sec_folder / out_filename

        # Skip if already exists
        if out_path.exists():
            pbar.update(1)
            continue

        try:
            if USE_GOOGLE_CLOUD and gcp_client:
                # Prepare request: supports SSML if needed (we use plain text now)
                input_text = texttospeech.SynthesisInput(text=tts_text)
                # voice selection: pick Sinhala WaveNet voice if available; otherwise pick language code
                voice = texttospeech.VoiceSelectionParams(language_code="si-LK", name=GCP_VOICE_NAME)
                audio_config = texttospeech.AudioConfig(
                    audio_encoding=GCP_AUDIO_ENCODING,
                    speaking_rate=GCP_SPEAKING_RATE,
                    pitch=GCP_PITCH
                )
                response = gcp_client.synthesize_speech(input=input_text, voice=voice, audio_config=audio_config)
                # response.audio_content is raw bytes (MP3)
                audio_bytes = response.audio_content
                # load into pydub for post-processing
                audio = AudioSegment.from_file(io.BytesIO(audio_bytes), format="mp3")
            else:
                # gTTS fallback
                # Use slow=False to let us control speed via pydub; we slow down after generation.
                tts = gTTS(text=tts_text, lang=LANG, slow=False)
                buf = io.BytesIO()
                tts.write_to_fp(buf)
                buf.seek(0)
                audio = AudioSegment.from_file(buf, format="mp3")

            # Post-process: slow a bit and increase volume
            processed = change_speed(audio, SLOW_FACTOR)
            processed = processed + VOLUME_GAIN_DB

            # Save final mp3
            save_audiosegment_to_mp3(processed, out_path)

        except Exception as e:
            # On error, print and attempt a minimal fallback filename using ascii-only
            print(f"Error generating audio for '{word}' in {sec}: {e}")
            fallback_out = sec_folder / f"{idx}_{ascii_name}.mp3"
            try:
                # Try gTTS again as last resort
                tts = gTTS(text=tts_text, lang=LANG, slow=False)
                buf = io.BytesIO(); tts.write_to_fp(buf); buf.seek(0)
                audio = AudioSegment.from_file(buf, format="mp3")
                processed = change_speed(audio, SLOW_FACTOR)
                processed = processed + VOLUME_GAIN_DB
                save_audiosegment_to_mp3(processed, fallback_out)
            except Exception as e2:
                print("Fallback also failed for", word, ":", e2)

        pbar.update(1)

pbar.close()
print("Done. Audio saved under:", OUTPUT_ROOT.resolve())
print("NOTE: To improve pronunciation for specific words, edit `pronunciation_overrides` and re-run the cell.")

c:\Users\94772\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
Generating audio: 100%|██████████| 160/160 [00:12<00:00, 12.66it/s]

Done. Audio saved under: C:\Users\94772\Desktop\HearingProject\hearing_project_v0.01\wiyanjana_audio_cloud
NOTE: To improve pronunciation for specific words, edit `pronunciation_overrides` and re-run the cell.
